In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# ==========================================
# 全局美化设置（主刊风格）
# ==========================================
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 1.5

# ==========================================
# 1. 配置文件路径
# ==========================================
file_path = "/mnt/e/2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/2-CR/mmc2.xlsx"  # 确保这里和你的文件名完全一致！
sheet_name = "Figure 6E"    # 你截图里的那个 Tab 页

print("⏳ 1. 正在读取顶刊 Excel 原始数据 (这可能需要十几秒)...")
# 不设表头读取，方便我们利用代码智能解析复杂的前三行
df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

print("🎯 2. 正在智能解析实验分组...")
# 提取前三行并转为小写/去除空格，防止格式误差
row_energy = df_raw.iloc[0].astype(str).str.strip().str.lower()
row_drug = df_raw.iloc[1].astype(str).str.strip().str.lower()
row_protein = df_raw.iloc[2].astype(str).str.strip() 

# 精准定位我们需要对比的 3 个核心列 (19% 蛋白质是标准鼠粮)
# 1. 对照组基线
col_ctrl = df_raw.columns[(row_energy == 'standard') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]
# 2. 热量限制组 (CR)
col_cr = df_raw.columns[(row_energy == 'low') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]
# 3. 雷帕霉素组 (Rapamycin)
col_rapa = df_raw.columns[(row_energy == 'standard') & (row_drug == 'rapamycin') & (row_protein.str.startswith('19'))][0]

print(f"✅ 列定位成功！\n   - 对照组(Control)在第 {col_ctrl} 列\n   - CR组(Low)在第 {col_cr} 列\n   - Rapa组在第 {col_rapa} 列")

# ==========================================
# 3. 数据提取与计算 Log2FC
# ==========================================
print("\n🧬 3. 正在搜索纯金靶点并计算逆转幅度...")
# 提取第一列的基因/蛋白名，全部转大写防报错
genes = df_raw.iloc[3:, 0].astype(str).str.upper() 

# 你的终极靶点候选池 (程序会把表里能找到的都画出来)
target_pool = ['HSPA8', 'CTSL', 'RPS10', 'RPS28', 'S100A6', 'LYZ2']
plot_data = []
found_genes = []

for gene in target_pool:
    idx = genes[genes == gene].index
    if len(idx) > 0:
        row_idx = idx[0]
        # 获取这三个组的定量数值
        val_ctrl = float(df_raw.iloc[row_idx, col_ctrl])
        val_cr = float(df_raw.iloc[row_idx, col_cr])
        val_rapa = float(df_raw.iloc[row_idx, col_rapa])
        
        # 计算 Fold Change (处理组 / 对照组) 并取 Log2
        if val_ctrl > 0:
            log2fc_cr = np.log2(val_cr / val_ctrl)
            log2fc_rapa = np.log2(val_rapa / val_ctrl)
            
            plot_data.append({'Protein': gene, 'Intervention': 'Caloric Restriction (CR)', 'Log2FC': log2fc_cr})
            plot_data.append({'Protein': gene, 'Intervention': 'Rapamycin', 'Log2FC': log2fc_rapa})
            found_genes.append(gene)
            print(f"   🌟 找到 {gene}: CR效应 = {log2fc_cr:.2f}, Rapa效应 = {log2fc_rapa:.2f}")

if not plot_data:
    print("❌ 警告：在这个表格里没有搜到目标基因。请检查靶点名是否在其他 Tab 页 (如 Figure 6C)。")
else:
    # ==========================================
    # 4. 绘制跨组学验证图
    # ==========================================
    print(f"\n🎨 4. 开始绘制包含 {len(found_genes)} 个靶点的机制验证图...")
    df_plot = pd.DataFrame(plot_data)
    
    fig, ax = plt.subplots(figsize=(1.5 * len(found_genes), 4.5), dpi=150) # 根据找到的基因数量动态调整宽度
    
    # 使用高级双色调
    sns.barplot(data=df_plot, x='Protein', y='Log2FC', hue='Intervention', 
                palette={'Caloric Restriction (CR)': '#4C72B0', 'Rapamycin': '#55A868'},
                edgecolor='black', linewidth=1.2, ax=ax)
    
    # 零刻度基准线
    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    
    # 标签与标题
    ax.set_title("Cross-omics Validation:\nProteomic Reversibility by mTOR Targeting", fontweight='bold', fontsize=14, pad=15)
    ax.set_ylabel("Protein Expression\n(Log2FC vs Baseline Control)", fontweight='bold', fontsize=12)
    ax.set_xlabel("")
    
    # 边框美化 (去顶去右)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(width=1.5, labelsize=12)
    
    plt.legend(title="", frameon=False, fontsize=10, loc='best')
    plt.tight_layout()
    
    # 保存图片
    plt.savefig("Figure_5G_Proteomics_Validation.pdf", bbox_inches='tight')
    plt.show()
    print("🎉 恭喜！完美的大图已生成并保存为 Figure_5G_Proteomics_Validation.pdf")

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# ==========================================
# 全局美化设置（主刊风格）
# ==========================================
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 1.5

# ==========================================
# 1. 配置文件路径 (适配你的最新保存路径)
# ==========================================
# 使用 raw string (r"") 防止 Windows 路径转义报错
file_path = "/mnt/e/2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR/table.xlsx"

print("⏳ 1. 正在读取独立的母表数据...")
# sheet_name=0 表示直接读取新建 Excel 里的第一个工作表
df_raw = pd.read_excel(file_path, sheet_name=0, header=None)

print("🎯 2. 正在智能解析实验分组...")
# 提取前三行并转为小写/去除空格，防止格式误差
row_energy = df_raw.iloc[0].astype(str).str.strip().str.lower()
row_drug = df_raw.iloc[1].astype(str).str.strip().str.lower()
row_protein = df_raw.iloc[2].astype(str).str.strip() 

try:
    # 精准定位我们需要对比的 3 个核心列 (19 代表标准蛋白质比例的鼠粮)
    # 1. 对照组基线
    col_ctrl = df_raw.columns[(row_energy == 'standard') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]
    # 2. 热量限制组 (CR)
    col_cr = df_raw.columns[(row_energy == 'low') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]
    # 3. 雷帕霉素组 (Rapamycin)
    col_rapa = df_raw.columns[(row_energy == 'standard') & (row_drug == 'rapamycin') & (row_protein.str.startswith('19'))][0]
    
    print(f"✅ 列定位成功！\n   - 对照组(Control)在第 {col_ctrl} 列\n   - CR组(Low)在第 {col_cr} 列\n   - Rapa组在第 {col_rapa} 列")
except IndexError:
    print("❌ 警告：未找到对应的分组列！请确认该表格向右滚动时，确实包含 Rapamycin 相关的列。")
    exit()

# ==========================================
# 3. 数据提取与计算 Log2FC
# ==========================================
print("\n🧬 3. 正在搜索纯金靶点并计算逆转幅度...")
# 提取第一列的基因/蛋白名，全部转大写防报错 (从第4行开始是数据)
genes = df_raw.iloc[3:, 0].astype(str).str.upper() 

8.74
# 你的终极靶点候选池 (涵盖翻译机器与蛋白质稳态)
target_pool = ['HSPA8', 'CTSL', 'RPS10', 'RPS28', 'S100A6']
plot_data = []
found_genes = []

for gene in target_pool:
    idx = genes[genes == gene].index
    if len(idx) > 0:
        row_idx = idx[0]
        # 获取这三个组的定量数值
        try:
            val_ctrl = float(df_raw.iloc[row_idx, col_ctrl])
            val_cr = float(df_raw.iloc[row_idx, col_cr])
            val_rapa = float(df_raw.iloc[row_idx, col_rapa])
            
            # 计算 Fold Change (处理组 / 对照组) 并取 Log2
            if val_ctrl > 0:
                log2fc_cr = np.log2(val_cr / val_ctrl)
                log2fc_rapa = np.log2(val_rapa / val_ctrl)
                
                plot_data.append({'Protein': gene, 'Intervention': 'Caloric Restriction (CR)', 'Log2FC': log2fc_cr})
                plot_data.append({'Protein': gene, 'Intervention': 'Rapamycin', 'Log2FC': log2fc_rapa})
                found_genes.append(gene)
                print(f"   🌟 找到 {gene}: CR效应 = {log2fc_cr:.2f}, Rapa效应 = {log2fc_rapa:.2f}")
        except ValueError:
            print(f"   ⚠️ {gene} 的数据存在空值或非数字，跳过计算。")

if not plot_data:
    print("❌ 警告：在这个表格里依然没有搜到目标基因。")
else:
    # ==========================================
    # 4. 绘制跨组学验证图
    # ==========================================
    print(f"\n🎨 4. 开始绘制包含 {len(found_genes)} 个靶点的机制验证图...")
    df_plot = pd.DataFrame(plot_data)
    
    # 动态图表宽度，保证柱子美观
    fig, ax = plt.subplots(figsize=(max(4, 1.5 * len(found_genes)), 4.5), dpi=150)
    
    # 使用高级双色调
    sns.barplot(data=df_plot, x='Protein', y='Log2FC', hue='Intervention', 
                palette={'Caloric Restriction (CR)': '#4C72B0', 'Rapamycin': '#55A868'},
                edgecolor='black', linewidth=1.2, ax=ax)
    
    # 零刻度基准线
    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    
    # 标签与标题
    ax.set_title("Cross-omics Validation:\nProteomic Reversibility by mTOR Targeting", fontweight='bold', fontsize=14, pad=15)
    ax.set_ylabel("Protein Expression\n(Log2FC vs Baseline Control)", fontweight='bold', fontsize=12)
    ax.set_xlabel("")
    
    # 边框美化 (去顶去右)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(width=1.5, labelsize=12)
    
    plt.legend(title="", frameon=False, fontsize=10, loc='best')
    plt.tight_layout()
    
    # 保存图片至同一文件夹
    save_path = "/mnt/e/2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR/Proteomics_Validation.pdf"
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f"🎉 恭喜！完美的大图已生成并保存至:\n👉 {save_path}")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# ==========================================
# 1. 注入 Nature / Cell 主刊级全局视觉规范
# ==========================================
sns.reset_orig()

# 核心字体设定
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'sans-serif']
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none' # 保留 AI 可编辑文本

# 极限字号微调 (主刊通常在 5pt - 8pt 之间)
mpl.rcParams['font.size'] = 6
mpl.rcParams['axes.titlesize'] = 7
mpl.rcParams['axes.labelsize'] = 7
mpl.rcParams['xtick.labelsize'] = 6
mpl.rcParams['ytick.labelsize'] = 6
mpl.rcParams['legend.fontsize'] = 5.5

# 极简纤细线条 (0.5pt 是主刊的灵魂)
mpl.rcParams['axes.linewidth'] = 0.5
mpl.rcParams['xtick.major.width'] = 0.5
mpl.rcParams['ytick.major.width'] = 0.5
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.direction'] = 'out'

sc_dpi = 300

# ==========================================
# 2. 路径配置 (3-CR 最新文件夹)
# ==========================================
file_path = "/mnt/e/2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR/table.xlsx"
output_dir = "/mnt/e/2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR"
os.makedirs(output_dir, exist_ok=True)

print("⏳ 正在读取母表数据并智能解析分组...")
df_raw = pd.read_excel(file_path, sheet_name=0, header=None)

row_energy = df_raw.iloc[0].astype(str).str.strip().str.lower()
row_drug = df_raw.iloc[1].astype(str).str.strip().str.lower()
row_protein = df_raw.iloc[2].astype(str).str.strip() 

try:
    col_ctrl = df_raw.columns[(row_energy == 'standard') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]
    col_cr = df_raw.columns[(row_energy == 'low') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]
    col_rapa = df_raw.columns[(row_energy == 'standard') & (row_drug == 'rapamycin') & (row_protein.str.startswith('19'))][0]
except IndexError:
    sys.exit("❌ 未找到对应的分组列，请检查 Excel。")

# ==========================================
# 3. 数据提取与计算 (终极防错版)
# ==========================================
genes = df_raw.iloc[3:, 0].astype(str).str.strip().str.upper() 
target_pool = ['HSPA8', 'CTSL', 'RPS10', 'RPS28', 'S100A6']
plot_data = []
found_genes = []

for gene in target_pool:
    idx = genes[genes == gene].index
    if len(idx) > 0:
        row_idx = idx[0]
        try:
            val_ctrl = float(df_raw.loc[row_idx, col_ctrl])
            val_cr = float(df_raw.loc[row_idx, col_cr])
            val_rapa = float(df_raw.loc[row_idx, col_rapa])
            
            pseudocount = 1e-6
            
            if val_ctrl > 0:
                log2fc_cr = np.log2((val_cr + pseudocount) / (val_ctrl + pseudocount))
                log2fc_rapa = np.log2((val_rapa + pseudocount) / (val_ctrl + pseudocount))
                
                plot_data.append({'Protein': gene, 'Intervention': 'CR', 'Log2FC': log2fc_cr})
                plot_data.append({'Protein': gene, 'Intervention': 'Rapamycin', 'Log2FC': log2fc_rapa})
                if gene not in found_genes: found_genes.append(gene)
        except ValueError:
            pass

if not plot_data:
    sys.exit("❌ 未搜到目标基因。")

# ==========================================
# 4. 主刊级图表渲染
# ==========================================
df_plot = pd.DataFrame(plot_data)

# ★ 设定绝对物理尺寸：宽 3.0 英寸 (约 76mm)，高 2.2 英寸 (约 55mm)
# 这是一张极其精致的小图，正好能塞进主刊排版的单栏中
fig, ax = plt.subplots(figsize=(3.0, 2.2), dpi=sc_dpi)

# 经典 NPG 配色：深海蓝 (#3C5488) 与 翡翠绿 (#00A087)
palette = {'CR': '#3C5488', 'Rapamycin': '#00A087'}

# 绘制柱状图，缩窄柱子宽度 (0.7) 留出呼吸感，边框线宽 0.5
sns.barplot(data=df_plot, x='Protein', y='Log2FC', hue='Intervention', 
            palette=palette, edgecolor='black', linewidth=0.5, 
            width=0.7, ax=ax, saturation=1.0)

# 0 刻度基准线，使用 0.5pt 的灰色虚线，避免抢夺数据本色的视觉焦点
ax.axhline(0, color='#555555', linewidth=0.5, linestyle='--', zorder=0)

# 精简的坐标轴标签
ax.set_ylabel(r"$\log_2$ FC (vs Control)", labelpad=4)
ax.set_xlabel("")

# 设置刻度长度
ax.tick_params(axis='both', length=2.5, pad=2)

# X轴基因名斜体处理 (符合基因/蛋白的标准书写规范的视觉感)
ax.set_xticklabels(ax.get_xticklabels(), fontstyle='italic')

# 极简边框 (仅保留左和下)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 主刊风格图例：放在图表上方外部，紧凑排列，无边框
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(0.0, 1.15), 
          ncol=2, frameon=False, handletextpad=0.4, columnspacing=1.0)

# 添加标题 (可选，主刊通常在图注里写标题，这里保留作为参考)
# ax.set_title("Proteomic validation of mTOR targets", pad=15)

plt.tight_layout()

# 双格式输出
save_pdf = os.path.join(output_dir, "Figure_5G_MainJournal_Style.pdf")
save_svg = os.path.join(output_dir, "Figure_5G_MainJournal_Style.svg")

plt.savefig(save_pdf, bbox_inches='tight')
plt.savefig(save_svg, bbox_inches='tight', format='svg')
plt.close(fig)

print(f"✅ 主刊级正图已生成！请将此 PDF/SVG 直接拖入 AI 感受其精致度：\n👉 {save_pdf}")